# Databricks Notebook: 10_bronze_ingest

**Purpose:** Landing-zone quality gate plus append-only load into Bronze.
Files that fail validation halt the run (intrusive); good files are appended with ingestion metadata.

### Target Tables:
- `bronze.events`, `bronze.users`, `bronze.catalog` (append-only; every delivery kept)
- `ops.ingestion_log` (one row per landing file per run: `LOADED` / `SKIPPED_UNCHANGED` / `REJECTED`)

### Ingestion Metadata Added:
- `_ingested_at`, `_source_type` (the event's `source` tag), `_source_file`, `_partition_dt`, `_batch_id`

In [ ]:
# %run ./00_config
# Placeholder Execution Flow:
# 1. Discover: list landing files; compare each SHA-256 with its last LOADED row in ops.ingestion_log.
#    New or changed -> load; unchanged -> SKIPPED_UNCHANGED.
# 2. Validate (plain Python): checksum vs _manifest.json, file parses, row count matches, required fields present.
#    Unknown new field -> SCHEMA_DRIFT warning. Any failure -> REJECTED, copy to _rejected/, halt before Silver.
# 3. Read with the explicit schema, FAILFAST, multiLine=true (events.json is currently one JSON array per file);
#    add _source_file, _partition_dt, _source_type, _batch_id, _ingested_at; append to bronze.events.
# 4. Append dims snapshots to bronze.users / bronze.catalog the same way.
# 5. Log every file in ops.ingestion_log; run anomaly checks (day volume vs 7-day median +-50%, missing dt, late files).
print("Bronze ingest notebook initialized.")
